# Create Embeddings for Labelled Dataset
---

Use Sentence Transfomers to create embeddings for all labelled emails, ready for modelling.


## Set Up
---

In [1]:
import pandas as pd

import joblib
import spacy

from sentence_transformers import SentenceTransformer

## Data Loading
----

In [2]:
labelled_df = pd.read_csv('data/labelled_emails.csv')

In [3]:
labelled_df.drop(columns=['categories'], inplace= True)

In [4]:
labelled_df = labelled_df[labelled_df['final_label'] != 'unknown'] # ignore unknown labels

In [5]:
nlp = spacy.load('en_core_web_sm')

In [6]:
def remove_names(text):
    text = str(text)
    doc = nlp(text)
    for e in reversed(doc.ents):
        if e.label_ in ("PERSON", "ORG", "DATE",): 
            text = text[:e.start_char] + text[e.start_char + len(e.text):]

    return text

In [7]:
labelled_df['email'] = labelled_df['email'].apply(remove_names)

## Get the Embeddings
----

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [9]:
embeddings = model.encode(
                labelled_df['email'].tolist(),
                batch_size=32,             
                show_progress_bar=True,    
                convert_to_numpy=True  
            )

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [10]:
embeddings.shape

(1019, 384)

In [11]:
joblib.dump(embeddings, "labelled_embeddings.pkl")

['labelled_embeddings.pkl']